In [1]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('2019-Nov.csv')

MemoryError: Unable to allocate 2.51 GiB for an array with shape (5, 67501979) and data type object

In [6]:
df.shape
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 67501979 entries, 0 to 67501978
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     str    
 1   event_type     str    
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  str    
 5   brand          str    
 6   price          float64
 7   user_id        int64  
 8   user_session   str    
dtypes: float64(1), int64(3), str(5)
memory usage: 4.5 GB


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [7]:
df.isnull().sum().sort_values(ascending=False)

category_code    21898171
brand             9224078
user_session           10
event_time              0
event_type              0
category_id             0
product_id              0
price                   0
user_id                 0
dtype: int64

In [4]:

# 1. Drop dữ liệu quan trọng
df = df.dropna(subset=['price', 'user_id'])

# 2. Xử lý brand
df['brand'] = df['brand'].fillna('unknown')
# 3. Xử lý category_code
df['category_code'] = df['category_code'].fillna('unknown')
df.isnull().sum()

event_time        0
event_type        0
product_id        0
category_id       0
category_code     0
brand             0
price             0
user_id           0
user_session     10
dtype: int64

In [5]:
# chuẩn hoá dữ liệu thời gian 
df['event_time'] = pd.to_datetime(df['event_time'])

In [10]:
# kiểm tra dữ liệu thời gian
df['event_time'].head()
df['event_time'].min(), df['event_time'].max()

(Timestamp('2019-11-01 00:00:00+0000', tz='UTC'),
 Timestamp('2019-11-30 23:59:59+0000', tz='UTC'))

In [6]:
# tạo thêm các cột giờ, ngày trong tuần, có phải cuối tuần không
df['hour']=df['event_time'].dt.hour
df['week_day']=df['event_time'].dt.weekday
df['is_weekend']=df['week_day'].isin([5,6])

In [12]:
df[['hour','weekday','is_weekend']].head()

KeyError: "['weekday'] not in index"

In [13]:
# xử lý giá trị bất thường
# kiểm tra xem giá trị price có phân bố như thế nào

df['price'].describe()

count    6.750198e+07
mean     2.924593e+02
std      3.556745e+02
min      0.000000e+00
25%      6.924000e+01
50%      1.657700e+02
75%      3.603400e+02
max      2.574070e+03
Name: price, dtype: float64

In [14]:
# có nhiều giá trị price bằng 0, có thể là lỗi dữ liệu hoặc sản phẩm miễn phí
(df['price'] == 0).sum()

np.int64(188088)

In [22]:
# xem xét price bằng 0 có trong các sự kiện nào 
df[df['price'] == 0]['event_type'].value_counts()


event_type
view    183680
cart      4408
Name: count, dtype: int64

In [7]:
# nếu price bằng 0 chủ yếu là view hoặc cart, có thể là lỗi dữ liệu, nên loại bỏ
df = df[df['price'] > 0]

In [24]:
# kiểm tra lại sau khi loại bỏ
df[df['price'] == 0]['event_type'].value_counts()

Series([], Name: count, dtype: int64)

In [25]:
# có nhiều giá trị price bằng 0, có thể là lỗi dữ liệu hoặc sản phẩm miễn phí
(df['price'] == 0).sum()

np.int64(0)

In [12]:
# check lại lỗi ở đâu
print("Min price:", df['price'].min())
print("Count price = 0:", (df['price'] == 0).sum())
print("Shape:", df.shape)

Min price: 0.77
Count price = 0: 0
Shape: (3122829, 9)


In [8]:
# tạo dữ liệu theo user 
user_data = df.groupby('user_id').agg({
    'event_type':'count',
    'price':'mean'
}).rename(columns={
    'event_type':'activity',
    'price':'avg_price'
})

purchase_data = df[df['event_type']=='purchase']

user_purchase = purchase_data.groupby('user_id').agg({
    'price':'count'
}).rename(columns={'price':'purchase_count'})

user_data = user_data.join(user_purchase, how='left').fillna(0)

In [9]:
user_data.head()


,activity,avg_price,purchase_count
user_id,,,
10300217,1,40.540000,0.0
29515875,11,275.157273,0.0
31198833,20,321.309000,0.0
34916060,1,295.940000,0.0
41798457,1,945.970000,0.0


In [10]:
user_data.describe()

,activity,avg_price,purchase_count
count,3.695107e+06,3.695107e+06,3.695107e+06
mean,1.821703e+01,3.069859e+02,2.481495e-01
std,4.525986e+01,3.158126e+02,1.334109e+00
min,1.000000e+00,7.700000e-01,0.000000e+00
25%,2.000000e+00,9.934437e+01,0.000000e+00
50%,5.000000e+00,2.052578e+02,0.000000e+00
75%,1.700000e+01,3.906725e+02,0.000000e+00
max,2.291900e+04,2.574070e+03,5.190000e+02


In [14]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904.0,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790.0,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266.0,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591.0,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683.0,313628f1-68b8-460d-84f6-cec7a8796ef2


In [ ]:
# lọc lại dữ liệu chỉ giữ lại 3 loại sự kiện: view, cart, purchase
df = df[df['event_type'].isin(['view','cart','purchase'])]
# phần này không cần cũng được

In [12]:
# Lưu dữ liệu sau tiền xử lý
df_clean = df.copy()

df_clean.to_csv('cleaned_ecommerce_data.csv', index=False)

print("Saved cleaned data successfully!") 
print("Shape:", df_clean.shape)

Saved cleaned data successfully!
Shape: (67313891, 12)


In [ ]:
# Đọc lại dữ liệu đã lưu để kiểm tra
import pandas as pd

df_clean = pd.read_csv('cleaned_ecommerce_data.csv')

print("Loaded cleaned data!")
df_clean.info()

Loaded cleaned data!
<class 'pandas.DataFrame'>
RangeIndex: 67313891 entries, 0 to 67313890
Data columns (total 12 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     str    
 1   event_type     str    
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  str    
 5   brand          str    
 6   price          float64
 7   user_id        int64  
 8   user_session   str    
 9   hour           int64  
 10  week_day       int64  
 11  is_weekend     bool   
dtypes: bool(1), float64(1), int64(5), str(5)
memory usage: 5.6 GB


In [ ]:
# kiểm tra lại dữ liệu sau khi lưu
print("Check price = 0:", (df_clean['price'] == 0).sum())   
print("Check missing:", df_clean.isnull().sum().sum())

Check price = 0: 0
Check missing: 10


In [16]:
# kiểm tra hành vi người dùng 
df_clean['event_type'].value_counts()

event_type
view        63372430
cart         3024522
purchase      916939
Name: count, dtype: int64

In [ ]:
# kiểm tra rồi chuyển sang phân tích tỷ lệ phần trăm    
df_clean['event_type'].value_counts(normalize=True)

event_type
view        0.941447
cart        0.044932
purchase    0.013622
Name: proportion, dtype: float64

In [17]:
# tính tỷ lệ chuyển đổi từ view -> cart -> purchase
counts = df_clean['event_type'].value_counts()

view = counts.get('view', 0)
cart = counts.get('cart', 0)
purchase = counts.get('purchase', 0)

cart_rate = cart / view
purchase_rate = purchase / cart

print(f"Cart rate: {cart_rate:.2%}")
print(f"Purchase rate: {purchase_rate:.2%}")

Cart rate: 4.77%
Purchase rate: 30.32%
